# RAG for the memoirs of Countess Schwerin

Source: `data/schwerin/lesefassung_2025.xml`, the reading edition of the digital edition (CC BY-NC 4.0). The text is in French.

**RAG** (retrieval-augmented generation) means: find the passages that best fit a question, then have a language model answer using only those passages.

How this notebook works:

1. Split the memoirs into paragraph-sized **chunks**.
2. Attach the people and places each chunk mentions, using the edition's registers, so that "mon mari" and "le comte de Schwerin" count as the same person.
3. Turn each chunk into a vector and store it in a searchable **index**.
4. For a question: translate it into French, collect candidates by meaning and by exact words, rerank them, and keep the best ones.
5. The language model answers from those passages and cites paragraph ids.

**How to use it:**

- `search("…", k=30)` lists the passages found, with scores, without writing an answer. Use it to check what retrieval finds.
- `ask("…")` writes an answer with citations. By default it reads 40 passages (`broad=True`), which suits topics spread across the whole memoirs; `broad=False` reads 20 and suits specific questions.
- Both accept `person=` and `place=` to limit the search to passages mentioning someone or somewhere; `show_entities("…")` finds the right id.
- Questions can be in any language. A full question works better than a single word, and naming synonyms helps (e.g. *grossesse, accouchement, couches*).

**Tested results:** two sections record checked runs: *Search results: pregnancy* (after Search) and *Checked results* (at the end). In short, citations are reliable, but paraphrases can shift meaning, so read the cited paragraph for important statements.

**To run it:** choose *Run > Run All Cells* and enter your DHInfra key when asked. The key is not saved in the notebook. If a cell runs for more than a few minutes, use *Kernel > Interrupt* and try a smaller `k` in `ask()`.


## Setup

These three cells install the packages, ask for your key, and connect to the models and the database.


In [2]:
# Installs the Python packages this notebook needs.
# You only need to run this once per environment.
# If imports still fail afterwards, restart the kernel and continue from the next cell.
%pip install langchain-openai langchain-chroma langchain-community langchain-text-splitters chromadb lxml pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from getpass import getpass

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

# Switches off LangSmith usage tracking, which this notebook does not use.
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["USER_AGENT"] = "schwerin_rag"

# The key is asked for at runtime so it never gets saved in this notebook.
if not os.environ.get("DHINFRA_API_KEY"):
    os.environ["DHINFRA_API_KEY"] = getpass("DHInfra API key: ")

DHInfra API key:  ········


In [4]:
# <-- FIX 2: Custom class to prevent the 422 "null content" error
# DHInfra rejects messages whose content is None.
# This small wrapper replaces None with an empty string before each request.
class SanitizedChatOpenAI(ChatOpenAI):
    def _get_request_payload(self, input_, *args, **kwargs):
        payload = super()._get_request_payload(input_, *args, **kwargs)
        if "messages" in payload:
            for msg in payload["messages"]:
                if msg.get("content") is None:
                    msg["content"] = ""
        return payload

# Initialize chat model using the sanitized class
# Connection to the language model. ask() below creates its own copy, answer_model.
model = SanitizedChatOpenAI(
    model="deepseek-v4.1-flash",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1",
    model_kwargs={"parallel_tool_calls": False}
)

# Initialize embedding model
# The embedding model turns a text into a list of numbers (a vector).
# Texts with similar meanings get similar vectors, which is what makes search possible.
embeddings = OpenAIEmbeddings(
    model="qwen3-embedding-8b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1"
)

# Initialize Chroma vector store
# Chroma is the database that stores each chunk with its vector and metadata.
vector_store = Chroma(
    collection_name="schwerin_memoirs",
    persist_directory="data/schwerin/chroma_schwerin",  # saved on disk, so it survives restarts
    embedding_function=embeddings
)

print("Chat model, embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done")

Chat model, embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done


## Chunk the TEI into paragraphs

Each `<p>` becomes one chunk. Paragraphs over 4000 characters are split. Editorial notes are left out of the text. Each chunk also records the register ids of the people and places it mentions.

### Load the registers

The text refers to the same person in many ways. The registers tell us who each TEI id (such as `P162`) really is. This cell must run before the chunking cell.


In [5]:
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path
from lxml import etree

TEI = "{http://www.tei-c.org/ns/1.0}"
XML_ID_ATTR = "{http://www.w3.org/XML/1998/namespace}id"

# The edition keeps its people and places in two separate TEI files.
# Here: where each file is saved locally, and where to download it.
REGISTER_FILES = {
    Path("data/schwerin/mgs_persons.xml"): "https://gams.uni-graz.at/o:mgs.persons/TEI_SOURCE",
    Path("data/schwerin/mgs_places.xml"): "https://gams.uni-graz.at/o:mgs.places/TEI_SOURCE",
}

# Download each register only if it is not already in data/schwerin/.
for path, url in REGISTER_FILES.items():
    if not path.exists():
        request = urllib.request.Request(url, headers={"User-Agent": "schwerin-rag-notebook"})
        with urllib.request.urlopen(request, timeout=60) as response:
            path.write_bytes(response.read())
        print(f"Downloaded {path}")

# Collects the text of all child elements with a given tag, without duplicates.
def distinct_text(parent, tag):
    values = []
    for element in parent.findall(TEI + tag):
        value = " ".join("".join(element.itertext()).split())
        if value and value not in values:
            values.append(value)
    return values

# REGISTER maps a TEI id (P162, O23) to its canonical name and details.
REGISTER = {}

# Build a readable name for each person, e.g. "Schwerin, Friedrich Wilhelm Graf von (1678-1727)".
for person in etree.parse("data/schwerin/mgs_persons.xml").getroot().iter(TEI + "person"):
    name = person.find(TEI + "persName")
    surname = " / ".join(distinct_text(name, "surname"))
    forename = " ".join(distinct_text(name, "forename"))
    roles = distinct_text(name, "roleName")
    # Drop a bare "von" when a fuller title exists, to avoid names like "Gräfin von von".
    roles = [role for role in roles if role != "von" or len(roles) == 1]
    given = " ".join(part for part in (forename, *roles) if part)
    birth, death = person.find(TEI + "birth"), person.find(TEI + "death")
    years = f"{(birth.get('when') if birth is not None else None) or '?'}-{(death.get('when') if death is not None else None) or '?'}"
    # Unnamed people (e.g. a governess) have no name in the register.
    label = ", ".join(part for part in (surname, given) if part) if surname or forename else ""
    if label and years != "?-?":
        label += f" ({years})"
    description = person.find(f"{TEI}note[@type='description']")
    REGISTER[person.get(XML_ID_ATTR)] = {
        "label": label,
        "description": " ".join("".join(description.itertext()).split()) if description is not None else "",
    }

# A place can have a modern and a historical name; keep both.
for place in etree.parse("data/schwerin/mgs_places.xml").getroot().iter(TEI + "place"):
    names = distinct_text(place, "placeName")
    REGISTER[place.get(XML_ID_ATTR)] = {"label": " / ".join(names), "description": ""}

# ALIASES records how the memoirs refer to each entity ("mon mari", "Vienne").
ALIASES = defaultdict(Counter)
# Each <rs> tag marks a mention in the memoirs and links it to one or more register ids.
# "mes enfants" links to every child, so one mention can yield several ids.
for mention in etree.parse("data/schwerin/lesefassung_2025.xml").getroot().iter(TEI + "rs"):
    surface = " ".join("".join(mention.itertext()).split())
    for ref in (mention.get("ref") or "").split():
        ALIASES[ref.split("#")[-1]][surface] += 1

def entity_label(entity_id):
    """Register name, or the memoir's own wording if the register has no name."""
    label = REGISTER.get(entity_id, {}).get("label", "")
    if label:
        return label
    if ALIASES[entity_id]:
        return f"«{ALIASES[entity_id].most_common(1)[0][0]}»"
    return entity_id

print(f"Register: {len(REGISTER)} entries; {len(ALIASES)} entities mentioned in the text")

Register: 750 entries; 734 entities mentioned in the text


In [6]:
from pathlib import Path
from copy import deepcopy
from lxml import etree
import pandas as pd

# ── Config ───────────────────────────────────────────────────
TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"

MEMOIR_FILE    = Path("data/schwerin/lesefassung_2025.xml")
MEMOIR_CSV     = Path("data/schwerin/schwerin_chunks.csv")

# The memoirs survive in two manuscripts, A and W. Where they differ, use A.
PREFERRED_WITNESS = "#A"   # apparatus reading to keep ("#A" or "#W")
# Paragraphs shorter than this are skipped (headings and fragments).
MIN_CHARS = 30             # skip fragments shorter than this
# Longer paragraphs are cut, since very long chunks embed less precisely.
MAX_CHARS = 4000           # split paragraphs longer than this
# Not used in this notebook: editorial notes are currently always left out.
INDEX_NOTES = True         # keep editorial notes as separate rows

# ── Shared helpers ───────────────────────────────────────────
def clean(s):
    """Normalize all whitespace (incl. internal newlines) to single spaces."""
    return " ".join(str(s).split())

def clean_text(elem):
    return " ".join(" ".join(elem.itertext()).split())

# <app> holds the variant readings of a passage. Keep just one, so the text reads continuously.
def resolve_apparatus(elem, witness=PREFERRED_WITNESS):
    """Replace each <app> with one chosen <rdg> reading."""
    for app in elem.findall(".//tei:app", TEI_NS):
        rdgs = app.findall("tei:rdg", TEI_NS)
        chosen = next(
            (r for r in rdgs if witness in (r.get("wit") or "")),
            rdgs[0] if rdgs else None
        )
        parent = app.getparent()
        if chosen is not None:
            chosen.tail = app.tail
            parent.replace(app, chosen)
        else:
            parent.remove(app)

# Takes notes and marginalia out of the running text.
# The text after each note stays; losing it was a bug in the earlier version.
def extract_notes(elem):
    """Remove <note> elements; return their cleaned texts."""
    notes = []
    for note in elem.findall(".//tei:note", TEI_NS):
        t = clean_text(note)
        if t:
            notes.append(t)
    # with_tail=False keeps the text after each note (lxml stores it in .tail)
    etree.strip_elements(elem, "{http://www.tei-c.org/ns/1.0}note", with_tail=False)
    return notes

# Cuts an over-long paragraph at sentence ends rather than mid-sentence.
def split_long(text, max_chars=MAX_CHARS):
    """Split oversized text at sentence-ish boundaries."""
    if len(text) <= max_chars:
        return [text]
    parts, current = [], ""
    for piece in text.replace(". ", ".\u0000").split("\u0000"):
        if current and len(current) + len(piece) + 1 > max_chars:
            parts.append(current.strip())
            current = piece
        else:
            current = f"{current} {piece}".strip()
    if current:
        parts.append(current.strip())
    return parts

# ════════════════════════════════════════════════════════════
# PART B: Memoirs (Lesefassung) — flat schema
# ════════════════════════════════════════════════════════════
# From here on, go through the memoirs and build one table row per chunk.
root = etree.parse(str(MEMOIR_FILE)).getroot()

memoir_rows = []

# Each TEI paragraph <p> is one unit of the text.
for p in root.findall(".//tei:text/tei:body//tei:p", TEI_NS):
    p_id = p.get(XML_ID) or f"P.unknown.{len(memoir_rows)}"
    # Work on a copy, so the removals below do not alter the parsed document.
    p = deepcopy(p)

    resolve_apparatus(p)
    extract_notes(p)          # notes dropped entirely
    text = clean_text(p)
    if len(text) < MIN_CHARS:
        continue

    # entities as mentioned in the kept text (deduplicated, order preserved)
    # The wording used in the text, e.g. "mon mari".
    def mentions(rs_type):
        seen, out = set(), []
        for r in p.findall(f".//tei:rs[@type='{rs_type}']", TEI_NS):
            name = clean_text(r)
            if name and name.lower() not in seen:
                seen.add(name.lower())
                out.append(name)
        return "; ".join(out)

    # canonical TEI ids, so "mon mari" and "comte de Schwerin" count as one person
    def entity_ids(rs_type):
        ids = []
        for r in p.findall(f".//tei:rs[@type='{rs_type}']", TEI_NS):
            for ref in (r.get("ref") or "").split():
                entity_id = ref.split("#")[-1]
                if entity_id not in ids:
                    ids.append(entity_id)
        return ids

    person_ids, place_ids = entity_ids("person"), entity_ids("place")

    # all plausible dates mentioned in this paragraph
    dates = []
    for d in p.findall(".//tei:date", TEI_NS):
        val = d.get("when-iso") or d.get("when") or ""
        if val[:4].isdigit() and 1680 <= int(val[:4]) <= 1740 and val not in dates:
            dates.append(val)

    # Usually one chunk per paragraph. Split paragraphs get ids like P.466.1, P.466.2.
    # A split paragraph's chunks all share the whole paragraph's people, places and dates.
    for j, chunk in enumerate(split_long(text)):
        suffix = f".{j+1}" if len(split_long(text)) > 1 else ""
        memoir_rows.append({
            "id": f"lesefassung:{p_id}{suffix}",
            "paragraph": p_id,
            "page_content": chunk,
            "persons": mentions("person"),
            "places": mentions("place"),
            "person_ids": ";".join(person_ids),
            "place_ids": ";".join(place_ids),
            "person_names": "; ".join(entity_label(e) for e in person_ids),
            "place_names": "; ".join(entity_label(e) for e in place_ids),
            "dates": "; ".join(dates),
            "source_file": MEMOIR_FILE.name,
        })

# Save as a CSV table, which you can open to inspect the chunks.
pd.DataFrame(memoir_rows).to_csv(MEMOIR_CSV, index=False)
print(f"Memoirs: {len(memoir_rows)} rows → {MEMOIR_CSV}")


Memoirs: 677 rows → data/schwerin/schwerin_chunks.csv


## Index

Embeds the chunks and saves them in `data/schwerin/chroma_schwerin`. Later runs reuse the saved index.

In [7]:
import pandas as pd
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.vectorstores.utils import filter_complex_metadata

# The table written by the chunking cell.
CSV_FILES = [Path("data/schwerin/schwerin_chunks.csv")]
# How many chunks are sent to the embedding model at once.
BATCH_SIZE = 100

# ── 1. Load CSVs into Documents ──────────────────────────────
docs = []
for csv_path in CSV_FILES:
    if not csv_path.exists():
        print(f"WARNING: {csv_path} not found, skipping.")
        continue
    df = pd.read_csv(csv_path).fillna("")
    for _, row in df.iterrows():
        meta = {k: str(row[k]) for k in df.columns if k != "page_content"}
        docs.append(Document(page_content=row["page_content"], metadata=meta))
    print(f"{csv_path.name}: {len(df)} rows loaded.")

print(f"Total: {len(docs)} documents.")

# Very short chunks (titles, date lines, single verses) match almost any query,
# so they stay in the CSV but are not searched.
MIN_INDEX_CHARS = 150
docs = [d for d in docs if len(d.page_content) >= MIN_INDEX_CHARS]
print(f"Searching {len(docs)} chunks; short fragments left out.")

# Set to True after changing how chunks are built: the check below compares ids only.
REBUILD_INDEX = False

# Compare what is saved with what the CSV contains, to avoid paying for embedding twice.
stored_ids = set(vector_store.get(include=[]).get("ids", []))
wanted_ids = {d.metadata["id"] for d in docs}

if stored_ids == wanted_ids and not REBUILD_INDEX:
    print(f"Index already holds all {len(stored_ids)} chunks; skipping embedding.")
else:
    # ── 2. Full reset of the store (clean slate) ─────────────────
    try:
        old_ids = vector_store.get().get("ids", [])
        if old_ids:
            for start in range(0, len(old_ids), 500):
                vector_store.delete(ids=old_ids[start:start+500])
            print(f"Deleted {len(old_ids)} old documents.")
    except Exception as e:
        print(f"Could not clear store (may be empty): {e}")

    # ── 3. Index in batches with stable IDs ──────────────────────
    # Chroma only accepts simple metadata values such as text and numbers.
    filtered = filter_complex_metadata(docs)
    ids = [d.metadata["id"] for d in filtered]
    assert len(ids) == len(set(ids)), "Duplicate IDs found — check the CSVs!"

    for start in range(0, len(filtered), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(filtered))
        vector_store.add_documents(
            documents=filtered[start:end],
            ids=ids[start:end],
        )
        print(f"Indexed {end}/{len(filtered)}")

    print("Indexing complete.")

schwerin_chunks.csv: 677 rows loaded.
Total: 677 documents.
Searching 653 chunks; short fragments left out.


/var/folders/zh/zrwd0srx1392p0_qgmc7q0s80000gn/T/ipykernel_62050/33812236.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores.utils import filter_complex_metadata


Deleted 677 old documents.
Indexed 100/653
Indexed 200/653
Indexed 300/653
Indexed 400/653
Indexed 500/653
Indexed 600/653
Indexed 653/653
Indexing complete.


## Entity lookup

Helpers for finding people and places by register name or by the memoirs' own wording.

- `show_entities("mon mari", kind="person")` lists everyone a phrase can refer to.
- `search(...)` and `ask(...)` accept `person=` and `place=`, as an id (`"P162"`) or an unambiguous name (`"Vienne"`).


In [8]:
import unicodedata
import pandas as pd

# Load the chunk table from the chunking cell.
chunks = pd.read_csv(MEMOIR_CSV).fillna("")

# ENTITY_CHUNKS maps an entity id to the ids of the chunks that mention it.
ENTITY_CHUNKS = defaultdict(list)
for _, row in chunks.iterrows():
    for entity_id in f"{row['person_ids']};{row['place_ids']}".split(";"):
        if entity_id:
            ENTITY_CHUNKS[entity_id].append(row["id"])

# Lower-cases and removes accents, so "Vienne" matches however it is typed.
def normalize(text):
    decomposed = unicodedata.normalize("NFKD", text.replace("’", "'"))
    return "".join(ch for ch in decomposed if not unicodedata.combining(ch)).casefold().strip()

def find_entities(text, kind=None):
    """Return matching entity ids, most frequently mentioned first.

    kind: "person", "place" or None. Exact matches win over partial ones and
    register names win over the wording used in the memoirs.
    """
    prefix = {"person": "P", "place": "O", None: ""}[kind]
    candidates = [e for e in ENTITY_CHUNKS if e.startswith(prefix)]
    if text in candidates:
        return [text]
    query = normalize(text)
    labels = {e: normalize(entity_label(e)) for e in candidates}
    aliases = {e: [normalize(a) for a in ALIASES[e]] for e in candidates}
    # Try the strictest kind of match first and stop at the first kind that finds anything.
    for matches in (
        [e for e in candidates if labels[e] == query],
        [e for e in candidates if query in aliases[e]],
        [e for e in candidates if query in labels[e]],
        [e for e in candidates if any(query in a for a in aliases[e])],
    ):
        if matches:
            return sorted(matches, key=lambda e: -len(ENTITY_CHUNKS[e]))
    return []

def show_entities(text, kind=None):
    """Look up who or what a name can refer to."""
    for entity_id in find_entities(text, kind):
        forms = ", ".join(a for a, _ in ALIASES[entity_id].most_common(4))
        print(f"{entity_id:6} {len(ENTITY_CHUNKS[entity_id]):4} chunks  {entity_label(entity_id)}  [{forms}]")

# Turns a name into exactly one id, and stops with a list of options if there are several.
def resolve_entity(text, kind):
    matches = find_entities(text, kind)
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise ValueError(f"No {kind} found for {text!r}.")
    options = "; ".join(f"{e} = {entity_label(e)}" for e in matches[:10])
    raise ValueError(f"{text!r} is ambiguous. Pass one of these ids: {options}")

def entity_filter(person=None, place=None):
    """Build a Chroma filter for chunks mentioning this person and/or place."""
    selected = [resolve_entity(value, kind) for value, kind in ((person, "person"), (place, "place")) if value]
    if not selected:
        return None
    for entity_id in selected:
        print(f"Filter: {entity_id} = {entity_label(entity_id)}")
    # If both a person and a place are given, keep chunks that mention both.
    allowed = set.intersection(*(set(ENTITY_CHUNKS[e]) for e in selected))
    if not allowed:
        raise ValueError("No chunk mentions all of the selected entities.")
    # This is the filter in the form Chroma expects: only these chunk ids may be searched.
    return {"id": {"$in": sorted(allowed)}}

# Example lookup, so you can see the output when the cell runs.
show_entities("Schwerin", kind="person")

P162    446 chunks  Schwerin, Friedrich Wilhelm Graf von (1678-1727)  [comte de Schwerin, le comte de Schwerin, mon mari, mon époux]
P154    101 chunks  Schwerin, Dorothea Luise Albertine Gräfin von (1713-1787)  [mes enfants, ma fille, les enfants, mes chers enfants]
P167     97 chunks  Schwerin, Ludwig Otto Sigismund Graf von (1710-1787)  [mes enfants, les enfants, mes chers enfants, enfants]
P159     93 chunks  Schwerin, Eugenius Graf von (1718-1769)  [mes enfants, les enfants, mes chers enfants, enfants]
P165     93 chunks  Schwerin, Leopold Ferdinand Graf von (1716-1757)  [mes enfants, les enfants, mes chers enfants, enfants]
P160     87 chunks  Schwerin, Friedrich Alexander Graf von (1714-1776)  [mes enfants, les enfants, mes chers enfants, enfants]
P82      44 chunks  Schwerin / Heyden, Luise Charlotte Gräfin von (1672-1748)  [ma belle-mère, Ma belle-mère, une très belle femme, sa sœur]
P42      43 chunks  Dohna-Schlobitten / Dönhoff / Schwerin, (Wilhelmine) Amalie Luise Gräfin v

## Retrieval

How a question finds its passages:

1. Translate the question into French.
2. Collect up to 40 chunks that are similar in meaning (embedding search).
3. Collect up to 40 chunks that contain the question's words (keyword search).
4. The reranker scores both piles together and keeps the best.

Run this cell before Search and Ask.


In [9]:
import math
import re
import requests
from langchain_core.messages import HumanMessage, SystemMessage

API_BASE = "https://api.dhinfra.uni-graz.at/v1"

answer_model = SanitizedChatOpenAI(
    model="deepseek-v4.1-flash",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base=API_BASE,
)

def to_french(question):
    """Translate the question, because the memoirs are in French."""
    reply = answer_model.invoke([
        SystemMessage(content=(
            "Translate the user's question into French. Keep names unchanged. "
            "If it is already French, repeat it. Reply with the question only."
        )),
        HumanMessage(content=question),
    ])
    return reply.content.strip()

# ── Exact-word search (the BM25 ranking method) ──────────────
# Common words that say nothing about a passage's topic.
STOPWORDS = set("""
a au aux avec ce ces comment dans de des du elle en est et il ils je la le les leur lui ma
mais me mes mon ne nous on ou par pas pour qu que quel quelle quels quelles qui sa se ses si
son sur ta te tes ton tu un une vos votre vous y comtesse decrit decrire dit parle
""".split())

def tokenize(text):
    """Split into lower-case words without accents, ignoring a final s (grossesse/grossesses)."""
    words = re.findall(r"[a-z]+", normalize(text))
    return [w[:-1] if len(w) > 3 and w.endswith("s") else w for w in words if w not in STOPWORDS]

doc_words = [Counter(tokenize(d.page_content)) for d in docs]
doc_lengths = [sum(words.values()) for words in doc_words]
average_length = sum(doc_lengths) / len(doc_lengths)
docs_with_word = Counter(word for words in doc_words for word in words)

def keyword_search(query, allowed_ids=None, k=40):
    """Return the chunks that best contain the query's words."""
    k1, b = 1.5, 0.75  # standard BM25 settings
    terms = set(tokenize(query))
    scored = []
    for doc, words, length in zip(docs, doc_words, doc_lengths):
        if allowed_ids is not None and doc.metadata["id"] not in allowed_ids:
            continue
        score = 0
        for term in terms & set(words):
            # Rare words count more; repeats count less and less.
            n = docs_with_word[term]
            rarity = math.log(1 + (len(docs) - n + 0.5) / (n + 0.5))
            score += rarity * words[term] * (k1 + 1) / (words[term] + k1 * (1 - b + b * length / average_length))
        if score > 0:
            scored.append((score, doc))
    scored.sort(key=lambda pair: -pair[0])
    return [doc for _, doc in scored[:k]]

def rerank(query, documents, top_n):
    """Score candidate chunks against the query; return the best as (chunk, score)."""
    response = requests.post(
        f"{API_BASE}/rerank",
        headers={"Authorization": f"Bearer {os.environ['DHINFRA_API_KEY']}"},
        json={
            "model": "qwen3-reranker-4b",
            "query": query,
            "documents": [d.page_content for d in documents],
            "top_n": top_n,
        },
        timeout=120,
    )
    response.raise_for_status()
    ranked = sorted(response.json()["results"], key=lambda r: -r["relevance_score"])
    return [(documents[r["index"]], r["relevance_score"]) for r in ranked]

def retrieve_with_scores(question, person=None, place=None, k=20, candidates=40):
    """Translate, collect candidates in two ways, and rerank them together."""
    query = to_french(question)
    print(f"Search query: {query}")
    chroma_filter = entity_filter(person, place)
    allowed_ids = set(chroma_filter["id"]["$in"]) if chroma_filter else None
    if allowed_ids is not None:
        chroma_filter = {"id": {"$in": sorted(allowed_ids & {d.metadata["id"] for d in docs})}}

    by_meaning = vector_store.similarity_search(query, k=candidates, filter=chroma_filter)
    by_words = keyword_search(query, allowed_ids, k=candidates)

    # Combine both piles; a chunk found by both is kept once.
    combined = list({d.metadata["id"]: d for d in by_meaning + by_words}.values())
    return rerank(query, combined, top_n=k) if combined else []

def retrieve(question, person=None, place=None, k=20, candidates=40):
    return [doc for doc, _ in retrieve_with_scores(question, person, place, k, candidates)]

def search(question, person=None, place=None, k=10):
    return retrieve_with_scores(question, person, place, k)

print("Retrieval ready.")

Retrieval ready.


## Search

Shows which passages are found, with their reranker scores, without waiting for a written answer. Any language works, because the query is translated into French first.


In [15]:
# Your search text, in any language. This cell shows what retrieval finds;
# no answer is written.
query = "Comment la comtesse vit-elle ses grossesses, ses accouchements, ses couches et ses fausses couches ?"  #change this

# Optional: limit the search to chunks mentioning a person and/or place.
# A name must identify one entity; use show_entities("...") to find its id.
results = search(query, person=None, place=None, k=30)

# Print the matches. The score comes from the reranker: higher is better.
for i, (doc, score) in enumerate(results, start=1):
    m = doc.metadata
    print(f"\n--- Result {i} (score: {round(score, 4)}) | {m.get('paragraph')} ---")
    print("Persons:", m.get("person_names", ""))
    print("Places: ", m.get("place_names", ""))
    print("Dates:  ", m.get("dates", ""))
    print(doc.page_content[:2000])

Search query: Comment la comtesse vit-elle ses grossesses, ses accouchements, ses couches et ses fausses couches ?

--- Result 1 (score: 0.5561) | P.268 ---
Persons: Braunschweig-Wolfenbüttel, Elisabeth Christine Herzogin von Kaiserin des Heiligen Römischen Reichs (1691-1750); Österreich, Maria Theresia Erzherzogin von (1717-1780); Schwerin, Friedrich Wilhelm Graf von (1678-1727)
Places:  Berlin; Wien
Dates:   1717-05
Sa Majesté l’Impératrice accoucha enfin le 13 de mai 1717 d’une archiduchesse dont la joie fut très grande à la vérité. Mais la perte n’était pas réparée, et malgré le contentement général on paraissait assez triste. Je commençais déjà à me remettre, quoique pas si bien qu’à Berlin . Je ne pouvais supporter la manière de veiller si tard, et d’être obligée de rester, et jouer dans le lit. C’est une mode introduite : dès ce qu’il y a une dame en couches, tout le monde y va jusqu’au soir fort tard , connu et inconnu. La dame en couches est au lit, à la vérité parée comme une

### Search results: pregnancy (16 November 2026)

Four searches with `search()` on her pregnancies, each checked against the text:

- **`"schwangerschaft"`, `k=10`:** all 10 passages are about her pregnancies, but every one contains the word *grossesse*, because the query was translated into that single word. Births, childbed and miscarriages, which the memoirs describe with other words (*enceinte*, *accouchement*, *en couches*, *fausse couche*), are missing. The memoirs contain roughly 90 passages on the topic.
- **Same query, `k=30`:** 29 of 30 passages are relevant, but they again almost all contain *grossesse*. Below result 10 the reranker scores fall under 0.06, so the extra hits are mostly passing mentions.
- **`"grossesse, enceinte, accouchement, en couches, fausse couche"`, `k=30`:** much better coverage. 19 of the 30 passages are new, including births (P.146, P.170), childbed (P.135, P.269, P.602), the second miscarriage (P.123) and her pregnancy during her flight (P.521, P.530). About 26 are about her own pregnancies; the rest concern other women, such as the Empress (P.268). The ranking is weaker: a list of words gives the reranker no question to judge against, scores are mostly below 0.01, and the best earlier hit (P.113) dropped out.
- **Full question with synonyms, `"Comment la comtesse vit-elle ses grossesses, ses accouchements, ses couches et ses fausses couches ?"`, `k=30`:** the best ranking. The top scores rise to 0.33–0.56, and the first 15 results are almost all strong hits, covering pregnancy (P.239, P.286), births (P.200, P.170), childbed (P.135, P.266, P.602) and both miscarriages (P.118, P.123). About 22 of 30 passages are about her own pregnancies. Weaknesses: the top hit (P.268) is the Empress's childbirth, not hers; P.363, P.366 and P.371 describe an illness with fever rather than a pregnancy; and P.113 and P.287 are still missing.

**Takeaway:** a single word finds precise but narrow results, and a list of synonyms finds more but ranks them poorly. A full question containing the synonyms ranks best, but no single search finds everything. For a complete overview of a frequent topic, combine several searches or use `ask(..., broad=True)`.


## Ask

Asks a question and gets an answer with paragraph citations. The answer prints first, then the paragraphs that were retrieved.

There are two modes:

- **Normal** (default): the model reads the best 20 passages. Good for specific questions, e.g. *"Wer ist Mme Wergeau?"*
- **Broad** (`broad=True`): the model reads the best 40 passages out of a larger pool and is told to collect everything relevant. Use it when a topic is spread across the whole memoirs, e.g. *"Wie beschreibt sie ihren Mann?"*. It takes longer.

The model only sees the retrieved passages. "Not found" therefore means not found *in those passages*, not that the memoirs never mention the topic.


In [11]:
SYSTEM = (
    "You answer questions about the memoirs of Countess Luise Charlotte von Schwerin "
    "(1684-1732), written in French. Use only the passages provided. After each claim, "
    "cite the paragraph id in square brackets, e.g. [P.5]. You only see a selection of "
    "the memoirs: if the passages do not contain the answer, say that you did not find "
    "it in the retrieved passages, and do not claim that the memoirs lack it. "
    "Answer in the language of the question."
)

# Extra instruction for broad questions, whose answer is spread over many passages.
BROAD_INSTRUCTION = (
    " The question concerns a topic that runs through the whole memoirs. Collect "
    "everything relevant from all passages, not just the first few, and group it by "
    "theme or by period of her life."
)

# How many passages each mode uses.
# k = how many passages the model reads.
# candidates = how many each search method collects for the reranker; it must be
# well above k, otherwise the reranker has nothing to choose from.
MODES = {
    "normal": {"k": 20, "candidates": 40},
    "broad": {"k": 40, "candidates": 80},
}

# The full RAG step: retrieve passages, then have the model answer from them alone.
# broad=True for topics spread over the whole memoirs. k and candidates override the mode;
# lower k (e.g. k=12) if answers are slow or hang.
def ask(question, person=None, place=None, broad=True, k=None, candidates=None):
    mode = MODES["broad" if broad else "normal"]
    k = k or mode["k"]
    candidates = candidates or mode["candidates"]
    hits = retrieve(question, person, place, k, candidates)
    # Give the model each passage with its paragraph id, so it can cite it,
    # and the real names, so it knows who "mon mari" is.
    context = "\n\n".join(
        f"[{d.metadata['paragraph']}]\n"
        f"People referred to: {d.metadata.get('person_names', '')}\n"
        f"Places referred to: {d.metadata.get('place_names', '')}\n"
        f"Text: {d.page_content}"
        for d in hits
    )
    system = SYSTEM + (BROAD_INSTRUCTION if broad else "")
    reply = answer_model.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"Passages:\n\n{context}\n\nQuestion: {question}"),
    ])
    print(reply.content)
    print(f"\nRetrieved ({len(hits)}):", ", ".join(d.metadata["paragraph"] for d in hits))

# Example question. Change the text to ask your own.
ask("Wie beschreibt die Gräfin ihre Erziehung?", broad=True)

# A specific question works with the normal mode:
# ask("Wer ist Mme Wergeau?")

# Limit to one person, using an id from show_entities:
# ask("Comment décrit-elle son mari ?", person="P162", broad=True)

Search query: Comment la comtesse décrit-elle son éducation ?
Die Gräfin beschreibt ihre Erziehung in den abgerufenen Passagen überwiegend als eine Erziehung durch familiäre Nachsicht, Schmeichelei und weltliche Anleitung; eine förmliche Schulbildung wird darin nicht geschildert. Später treten Selbstbildung, religiöse Unterweisung und Erziehungskonflikte als Mutter hinzu.

- **Frühe Kindheit – Verwöhnung statt Zucht:** Nach dem Tod der Mutter behält der Vater sie zunächst, weil sie ihm ähnlich sieht, und gibt sie erst mit zweieinhalb Jahren zur Tante. Dort wird sie von Tante, deren Schwester und Schwager, die keine Kinder haben, geradezu vergöttert; sie beschreibt sich als verzogenes Kind, als „dispensatrice des grâces“, und sagt, bei Bestrafung habe sich das ganze Haus bewegt, um ihr Ersatzvergnügen zu verschaffen. Diese „blinde Zärtlichkeit“ habe sie mit Schmeichelei, Süße und Gefälligkeit genährt, was sie als gefährliches Gift bezeichnet [P.7].

- **Französische Gouvernante und elte

## Checked results (16 November 2026)

Question: *"Wie beschreibt die Gräfin ihre Erziehung?"*, asked with `broad=True` (40 passages). Each statement in the answer was compared with the paragraph it cites.

**Overall:** a very good answer: clearly structured, concrete and sourced 

- **Grouped by theme and life stage:** early childhood with the aunt, the French governess, the stepmother, meeting her future husband, the Vienna salon, religious instruction and later conflicts over her own children's upbringing. This gives a readable overview of a topic spread across the whole memoirs.
- **Concrete statements:** each point says who did what and when (e.g. aged two and a half, five, eight)
- **Cited throughout:** almost every statement carries a paragraph id, so it can be checked directly in the text.
- **Close to the source:** key French phrases are quoted ("dispensatrice des grâces", "restauratrice de la vertu", "belles leçons d'un mari à une jeune femme")
- **Honest about limits:** it says that formal schooling is not described in the retrieved passages instead of inventing it.
- **Uses scattered passages:** besides the childhood chapters it draws on much later passages (P.224, P.253, P.460, P.482).

**Correct:** every paragraph id matches its statement and no quotation is invented. The core passages (P.7, P.8, P.11, P.18, P.72) are ranked at the top.

**Errors and inaccuracies:**
- **P.11, imprecise:** "suite pernicieuse d'une éducation oisive" refers to her boredom with country life, which she calls a harmful consequence of an idle upbringing. The answer applies the phrase to the country upbringing itself.
- **P.332, overstated:** she does not seek out Père Brean; Countess Batthyány suggests him, and she hesitates for fear of being found out.
- **P.403, shifted:** "Todesmut" in the answer; the text says martyrdom would be "le comble de mon bonheur".
- **Unsourced:** the closing sentence contrasting her own upbringing with her children's is the model's interpretation.

**Content:**
- About a third of the answer is about her conversion, which does not answer a question about her upbringing. The broad-mode instruction ("collect everything relevant") encourages this.
- P.5, where she sums up her own upbringing ("une jeune personne élevée dans tout ce qui peut plaire à son ambition et à ses sens"), was retrieved but not used.
- The last of the 40 passages (Vienna, money) are off-topic; for this question about 15 to 20 would have been enough.

**Takeaway:** citations are reliable, but paraphrases can shift meaning. For important statements, read the cited paragraph.
